# Lesson 9 — Constrained NLP, KKT, IPOPT

## Learning objectives

1. State the KKT conditions for an inequality-constrained NLP.
2. Recognize constraint qualifications and what they buy you.
3. Walk through the interior-point barrier method conceptually.
4. Use `discopt`'s IPOPT wrapper {cite:p}`Wachter2006` and read its output.

## 1. KKT conditions

For

$$\min f(x) \;\;\text{s.t.}\;\; g_i(x) \le 0,\; h_j(x) = 0,$$

a *local* min $x^\star$ satisfying a constraint qualification (CQ) admits multipliers $\lambda \ge 0, \mu$ with:

$$
\begin{aligned}
\nabla f + \sum_i \lambda_i \nabla g_i + \sum_j \mu_j \nabla h_j &= 0 & \text{(stationarity)} \\
g_i(x^\star) \le 0,\; h_j(x^\star) = 0 & & \text{(primal feasibility)} \\
\lambda_i \ge 0 & & \text{(dual feasibility)} \\
\lambda_i g_i(x^\star) = 0 & & \text{(complementary slackness)}
\end{aligned}
$$

For convex problems, KKT is *sufficient* — any KKT point is a global optimum {cite:p}`Boyd2004`.

## 2. Constraint qualifications

CQs ensure KKT is necessary. Common ones:

- **LICQ:** active constraint gradients are linearly independent.
- **MFCQ:** there exists a direction strictly improving every active inequality.
- **Slater's condition** (convex): there's a strictly feasible point.

Without a CQ, the multipliers may not exist (or may be unbounded) at $x^\star$ {cite:p}`Nocedal2006`.

## 3. Interior-point methods (IPM)

Reformulate inequalities with a logarithmic barrier:

$$\min f(x) - \mu \sum_i \log(-g_i(x)) \;\;\text{s.t.}\;\; h(x) = 0$$

Solve the equality-constrained problem for decreasing $\mu \to 0$. Each subproblem is solved with Newton on the KKT system. This is the heart of IPOPT {cite:p}`Wachter2006`.

## 4. The KKT linear system

At each Newton step:

$$\begin{bmatrix} W_k & A_k^\top \\ A_k & 0 \end{bmatrix} \begin{bmatrix} \Delta x \\ \Delta \lambda \end{bmatrix} = -\begin{bmatrix} \nabla L_k \\ c_k \end{bmatrix}$$

with $W_k$ the Hessian of the Lagrangian and $A_k$ the constraint Jacobian. Sparse direct solvers (MA27, MUMPS, HSL_MA86) are the bottleneck for large problems.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model

# Constrained NLP: min (x-1)^2 + (y-2)^2  s.t.  x^2 + y^2 <= 1.
# Algebraically: KKT predicts x*=1/(1+lam), y*=2/(1+lam), with the
# constraint active: 1/(1+lam)^2 + 4/(1+lam)^2 = 1  -->  lam = sqrt(5)-1.
m = Model("circ")
x = m.continuous("x", lb=-2, ub=2)
y = m.continuous("y", lb=-2, ub=2)
m.subject_to(x**2 + y**2 <= 1)
m.minimize((x - 1)**2 + (y - 2)**2)
r = m.solve()

print("x*, y* =", r.value(x), r.value(y))
print("f*     =", round(r.objective, 6))
print("on the unit circle? x^2 + y^2 =", round(r.value(x)**2 + r.value(y)**2, 6))

# discopt exposes the KKT multipliers passed back by IPOPT in
# r.constraint_duals (keyed by Constraint.name). The single inequality here
# is named "c0", so:
lam = float(r.constraint_duals["c0"])
print(f"discopt lambda* = {lam:.6f}   (algebraic: sqrt(5) - 1 = {np.sqrt(5) - 1:.6f})")
# Exercise 1 derives this same lambda by hand from the KKT system; comparing
# the two is the cleanest way to convince yourself the numerics are right.


## 5. KKT in the small example

For our problem, $\nabla f = (2(x-1), 2(y-2))$, $\nabla g = (2x, 2y)$, so stationarity gives

$$(2(x-1), 2(y-2)) + \lambda (2x, 2y) = 0,$$

i.e., $x = 1/(1+\lambda)$, $y = 2/(1+\lambda)$. Combined with $x^2 + y^2 = 1$ (active): $5/(1+\lambda)^2 = 1$, so $\lambda = \sqrt 5 - 1 \approx 1.236$. Verify against the solver's multiplier.

## 6. Why the barrier works

The log-barrier $-\mu \log(-g)$ blows up at the boundary, keeping iterates strictly feasible. As $\mu \to 0$, the central path converges to the constrained optimum. Polynomial-time complexity (in the conic case) is the great theoretical achievement of {cite:p}`Karmarkar1984` and successors {cite:p}`Wright1997,Nocedal2006`.

## References
{cite:p}`Nocedal2006` Ch 12, 19; {cite:p}`Wachter2006` for IPOPT itself; {cite:p}`Boyd2004` for the convex case.